In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

In [3]:
PDF_PATH = pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "gemma2:27b-instruct-q4_K_M"

# Ontology modes:
#   None       -> no ontology
#   "minimal"  -> minimal fallback ontology
#   path       -> real ontology file
SEED_ONTOLOGY_INPUT = str("../data/ontology/ContextOntology-COInd4.owl")

#### Layer 1

In [ ]:
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf
from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend
from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer

pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
model_name = "gemma2:27b-instruct-q4_K_M"

raw_text = extract_text_from_pdf(pdf_path)

document = Document(
    doc_id="pdf_test_001",
    source_path=pdf_path,
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level="Treat machine types, failure types, and symptom types as concepts; treat document-specific occurrences as individuals.",
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=model_name,
    user_guidance=guidance,
)

ollama = OllamaBackend()

translator = DeepTranslatorBackend(max_chars=4000)

pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,          # enable translation
            translator=translator,   # use free translator
            source_language="fr",    # for example
            target_language="en",
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=2,
            temperature=0.0,
        ),
    ]
)


runner = Runner(pipeline)
final_state = runner.run(state)

final_state.logs

['[layer00_preprocessing] started',
 '[layer00_preprocessing] translation applied',
 '[layer00_preprocessing] produced 27 chunks',
 '[layer00_preprocessing] finished',
 '[layer01_linguistic_expression_extraction] started',
 '[layer01_linguistic_expression_extraction] extracted 15 unique expressions',
 '[layer01_linguistic_expression_extraction] finished']

In [4]:
len(final_state.document.chunks)

27

In [5]:
for expr in final_state.linguistic_expressions[:10]:
    print(expr)

LinguisticExpression(expr_id='expr_00000', text='alarms', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=40, chunk_end_char=46, doc_start_char=40, doc_end_char=46, snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main cat')], confidence=None)
LinguisticExpression(expr_id='expr_00001', text='messages', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=51, chunk_end_char=59, doc_start_char=51, doc_end_char=59, snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main categories: 1 al')], confidence=None)
LinguisticExpression(expr_id='expr_00002', text='immediate and controlled stop', label='operational state', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=554, chunk_end_char=583, doc_start_char=554, doc

#### Layer 3

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# NeoOLAF imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
PDF_PATH = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = "gemma2:27b-instruct-q4_K_M"

# Debug limits
MAX_CHUNKS = 2
MAX_EXPRESSIONS = 5

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# Optional semantic guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

# ---------------------------------------------------------
# State and backends
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect one enriched expression
# ---------------------------------------------------------
item = final_state.enriched_expressions[0]

print("Base expression:", item.base_expression.text)
print("Aliases:", item.aliases)
print("Alias sources:")
pprint(item.alias_sources)

print("\nSynonyms:", item.synonyms)
print("Synonym sources:")
pprint(item.synonym_sources)

print("\nLexical variants:", item.lexical_variants)
print("Lexical variant sources:")
pprint(item.lexical_variant_sources)

print("\nDefinition:", item.definition)
print("Ontology hints:", item.ontology_hints)

print("\nEvidence sources:")
for ev in item.enrichment_evidence:
    print("-", ev.source, "|", ev.reference)

Base expression: alarms
Aliases: ['alarm', 'dismay', 'consternation', 'warning device', 'alarm system', 'alert', 'warning signal', 'alarum', 'alarm clock', 'appal', 'appall', 'horrify', 'Alarm device']
Alias sources:
{'Alarm device': ['wikipedia', 'llm'],
 'alarm': ['wordnet'],
 'alarm clock': ['wordnet'],
 'alarm system': ['wordnet'],
 'alarum': ['wordnet'],
 'alert': ['wordnet'],
 'appal': ['wordnet'],
 'appall': ['wordnet'],
 'consternation': ['wordnet'],
 'dismay': ['wordnet'],
 'horrify': ['wordnet'],
 'warning device': ['wordnet'],
 'warning signal': ['wordnet']}

Synonyms: ['alarm', 'dismay', 'consternation', 'warning device', 'alarm system', 'alert', 'warning signal', 'alarum', 'alarm clock', 'appal', 'appall', 'horrify']
Synonym sources:
{'alarm': ['wordnet'],
 'alarm clock': ['wordnet'],
 'alarm system': ['wordnet'],
 'alarum': ['wordnet'],
 'alert': ['wordnet', 'llm'],
 'appal': ['wordnet'],
 'appall': ['wordnet'],
 'consternation': ['wordnet'],
 'dismay': ['wordnet'],
 'hor

In [26]:
final_state.enriched_expressions

[EnrichedExpression(base_expression=LinguisticExpression(expr_id='expr_00000', text='alarms', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=-1, chunk_end_char=-1, doc_start_char=-1, doc_end_char=-1, snippet='SPRINT32linear-FA-MP-v0.1-fr Page 8-4 8 Alarmes et messages 8.1 Introduction Les alarmes et les messages sont divisés en deux catégories principales : 1 les alarmes et messages de la CNC, pour lesquels nous renvoyons au ma- nuel d’entretien de la CNC Fanuc série ........ 2 les alarmes et messages de')], confidence=None), aliases=['alarm', 'alert'], synonyms=['warning signal', 'warning device'], lexical_variants=[], definition='a signal indicating a problem or condition requiring urgent attention.', ontology_hints=['failure symptom', 'observed-by: sensors'], enrichment_evidence=[EnrichmentEvidence(source='wordnet', content='fear resulting from the awareness of danger', reference=None), EnrichmentEvidence(source='wordnet', conte

#### Layer 4

In [7]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)
from neoolaf.layers.layer03_candidate_typing_resolution.component import (
    CandidateTypingResolutionLayer,
)
from neoolaf.layers.layer04_candidate_relation_extraction.component import (
    CandidateRelationExtractionLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = None#20

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 4 outputs
# ---------------------------------------------------------
print("Candidate relation assertions:", len(final_state.candidate_relation_assertions))

for item in final_state.candidate_relation_assertions[:10]:
    print("=" * 80)
    print("Assertion ID:", item.assertion_id)
    print("Relation:", item.relation_label, "|", item.relation_candidate_id)
    print("Source:", item.source_candidate_label, "|", item.source_candidate_id, "|", item.source_candidate_type)
    print("Target:", item.target_candidate_label, "|", item.target_candidate_id, "|", item.target_candidate_type)
    print("Chunk:", item.chunk_id)
    print("Confidence:", item.confidence)
    print("Justification:", item.justification)

Candidate relation assertions: 7
Assertion ID: rel_assert_00000
Relation: emitted by | cand_r_00000
Source: PLC | cand_e_00002 | entity
Target: message | cand_e_00000 | entity
Chunk: chunk_0000
Confidence: 0.95
Justification: The text states that messages are emitted by the PLC.
Assertion ID: rel_assert_00002
Relation: compromises | cand_r_00001
Source: PLC | cand_e_00002 | entity
Target: reset | cand_s_00000 | event
Chunk: chunk_0000
Confidence: 0.85
Justification: The text states that messages emitted by the PLC can indicate a problem that may compromise the proper functioning of the machine and its machining cycle. These messages can trigger an immediate controlled stop, which is reset by pressing the RESET PRG button.
Assertion ID: rel_assert_00003
Relation: machine | cand_r_00002
Source: PLC | cand_e_00002 | entity
Target: cycle | cand_e_00003 | entity
Chunk: chunk_0000
Confidence: 0.85
Justification: The text states that messages from the PLC indicate a problem that can compromis

#### Layer 5

In [5]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)
from neoolaf.layers.layer03_candidate_typing_resolution.component import (
    CandidateTypingResolutionLayer,
)
from neoolaf.layers.layer04_candidate_relation_extraction.component import (
    CandidateRelationExtractionLayer,
)
from neoolaf.layers.layer05_candidate_triple_generation.component import (
    CandidateTripleGenerationLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = None#20
MAX_ASSERTIONS = None

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=MAX_ASSERTIONS,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(
    pipeline=pipeline,
    runs_root="runs",
    verbose=True,
)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 5 outputs
# ---------------------------------------------------------
print("Candidate triples:", len(final_state.candidate_triples))

for triple in final_state.candidate_triples[:10]:
    print("=" * 90)
    print("Triple ID:", triple.triple_id)
    print("S:", triple.subject_label, "|", triple.subject_id, "|", triple.subject_type)
    print("P:", triple.predicate_label, "|", triple.predicate_id)
    print("O:", triple.object_label, "|", triple.object_id, "|", triple.object_type)
    print("Chunk:", triple.chunk_id)
    print("Confidence:", triple.confidence)
    print("Justification:", triple.justification)

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[NeoOLAF] Run directory: runs\run_20260317_131903
[NeoOLAF] Pipeline started with 6 layers
[NeoOLAF] Layer 0/5: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/5: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 385.61s
[NeoOLAF] Layer 2/5: layer02_candidate_enrichment

[NeoOLAF] Starting layer: layer02_candidate_enrichment


[NeoOLAF] Finished layer: layer02_candidate_enrichment in 889.29s
[NeoOLAF] Layer 3/5: layer03_candidate_typing_resolution

[NeoOLAF] Starting layer: layer03_candidate_typing_resolution


[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 483.40s
[NeoOLAF] Layer 4/5: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction


[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 320.75s
[NeoOLAF] Layer 5/5: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.10s
[NeoOLAF] Pipeline finished in 2079.17s
[NeoOLAF] Total run time: 2079.17s
Candidate triples: 7
Triple ID: triple_00000
S: PLC | cand_e_00002 | entity
P: emitted by | cand_r_00000
O: message | cand_e_00000 | entity
Chunk: chunk_0000
Confidence: 0.95
Justification: The text states that messages are emitted by the PLC.
Triple ID: triple_00001
S: PLC | cand_e_00002 | entity
P: compromises | cand_r_00001
O: reset | cand_s_00000 | event
Chunk: chunk_0000
Confidence: 0.85
Justification: The text states that messages emitted by the PLC can indicate a problem that may compromise the proper functioning of the machine and its machining cycle. These messages can trigger an immediate controlled stop, which is reset by pressing the RESET PRG button.
Triple ID: triple_00002
S: PLC | cand_e_00002 | entity
P: machine | cand_r_00002
O: cycle | cand_e_00003 | entity
Chunk: chunk_0000
Confidence: 0.85
Justification: The text states th

In [8]:
print(len(final_state.document.chunks))
print(len(final_state.linguistic_expressions))
print(len(final_state.enriched_expressions))
print(len(final_state.entity_candidates))
print(len(final_state.relation_candidates))
print(len(final_state.attribute_candidates))
print(len(final_state.event_candidates))

31
26
20
7
9
0
3


#### Layer 6

In [5]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 6 outputs
# ---------------------------------------------------------
print("Concept candidates:", len(final_state.concept_candidates))
print("Ontology relation candidates:", len(final_state.ontology_relation_candidates))

if final_state.concept_candidates:
    print("\nFirst concept candidate:")
    pprint(final_state.concept_candidates[0])

if final_state.ontology_relation_candidates:
    print("\nFirst ontology relation candidate:")
    pprint(final_state.ontology_relation_candidates[0])

[NeoOLAF] Run directory: runs\run_20260319_153949
[NeoOLAF] Pipeline started with 7 layers
[NeoOLAF] Layer 0/6: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/6: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 457.86s
[NeoOLAF] Layer 2/6: layer02_candidate_enrichment

[NeoOLAF] Starting layer: layer02_candidate_enrichment


[NeoOLAF] Finished layer: layer02_candidate_enrichment in 861.20s
[NeoOLAF] Layer 3/6: layer03_candidate_typing_resolution

[NeoOLAF] Starting layer: layer03_candidate_typing_resolution


KeyboardInterrupt: 

#### Layer 7

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 7 outputs
# ---------------------------------------------------------
print("Concept hierarchy links:", len(final_state.concept_hierarchy_links))
print("Relation hierarchy links:", len(final_state.relation_hierarchy_links))

if final_state.concept_hierarchy_links:
    print("\nFirst concept hierarchy link:")
    pprint(final_state.concept_hierarchy_links[0])

if final_state.relation_hierarchy_links:
    print("\nFirst relation hierarchy link:")
    pprint(final_state.relation_hierarchy_links[0])

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[NeoOLAF] Run directory: runs\run_20260319_161251
[NeoOLAF] Pipeline started with 8 layers
[NeoOLAF] Layer 0/7: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/7: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 546.58s
[NeoOLAF] Layer 2/7: layer02_candidate_enrichment

[NeoOLAF] Starting layer: layer02_candidate_enrichment


[NeoOLAF] Finished layer: layer02_candidate_enrichment in 1229.40s
[NeoOLAF] Layer 3/7: layer03_candidate_typing_resolution

[NeoOLAF] Starting layer: layer03_candidate_typing_resolution


Layer 3 - typing:   0%|          | 0/20 [00:00<?, ?it/s]

#### Layer 8

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 8 outputs
# ---------------------------------------------------------
print("Axiom schema candidates:", len(final_state.axiom_schema_candidates))

if final_state.axiom_schema_candidates:
    print("\nFirst axiom schema candidate:")
    pprint(final_state.axiom_schema_candidates[0])

[NeoOLAF] Run directory: runs\run_20260319_171031
[NeoOLAF] Pipeline started with 9 layers
[NeoOLAF] Layer 0/8: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/8: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


Layer 1 - chunks:   0%|          | 0/2 [00:00<?, ?it/s]

#### Layer 9

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 9 outputs
# ---------------------------------------------------------
print("General axiom candidates:", len(final_state.general_axiom_candidates))

if final_state.general_axiom_candidates:
    print("\nFirst general axiom candidate:")
    pprint(final_state.general_axiom_candidates[0])

[NeoOLAF] Run directory: runs\run_20260319_171353
[NeoOLAF] Pipeline started with 10 layers
[NeoOLAF] Layer 0/9: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/9: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


Layer 1 - chunks:   0%|          | 0/2 [00:00<?, ?it/s]

#### Layer 10

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 2
MAX_EXPRESSIONS = 20
MAX_RELATION_MENTIONS = 20

# ---------------------------------------------------------
# Document and state
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
    ],
    verbose=True,
)

runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect Layer 10 outputs
# ---------------------------------------------------------
print("Validation report:")
pprint(final_state.validation_report)

print("\nReasoning report notes:")
pprint(final_state.reasoning_report.notes if final_state.reasoning_report else [])

print("\nNumber of inferred triples:")
print(len(final_state.reasoning_report.inferred_triples) if final_state.reasoning_report else 0)

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[NeoOLAF] Run directory: runs\run_20260319_171731
[NeoOLAF] Pipeline started with 11 layers
[NeoOLAF] Layer 0/10: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.01s
[NeoOLAF] Layer 1/10: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


Layer 1 - chunks:   0%|          | 0/2 [00:00<?, ?it/s]

#### Layer 11 + 12

In [7]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.ontology.factory import build_seed_ontology

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer
from neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer
from neoolaf.layers.layer12_serialization.component import SerializationLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

MAX_CHUNKS = 1 #2
MAX_EXPRESSIONS = 2 #20
MAX_RELATION_MENTIONS = 2 #20

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# User guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

# ---------------------------------------------------------
# Seed ontology
# ---------------------------------------------------------
seed_ontology = build_seed_ontology(SEED_ONTOLOGY_INPUT)

print("Seed ontology loaded:", seed_ontology is not None)
if seed_ontology is not None:
    print("Ontology URI:", seed_ontology.ontology_uri)
    print("Ontology label:", seed_ontology.ontology_label)
    print("Classes:", len(seed_ontology.get_classes()))
    print("Properties:", len(seed_ontology.get_properties()))

# ---------------------------------------------------------
# Pipeline state
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
    seed_ontology=seed_ontology,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=ollama,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=ollama,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=ollama,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=ollama,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=ollama,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
        InferenceCompletionLayer(
            max_inferred_triples=None,
            verbose=True,
        ),
        SerializationLayer(
            output_subdir="data/exports",
            base_uri="http://neoolaf.org/resource/",
            verbose=True,
        ),
    ],
    verbose=True,
)

# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
runner = Runner(pipeline=pipeline, runs_root="runs", verbose=True)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect outputs
# ---------------------------------------------------------
print("Completion candidates:", len(final_state.completion_candidates))
if final_state.completion_candidates:
    print("\nFirst completion candidate:")
    pprint(final_state.completion_candidates[0])

print("\nExport folder:")
export_dir = Path(final_state.artifact_dir) / "data" / "exports"
print(export_dir)

if export_dir.exists():
    print("\nExported files:")
    for path in sorted(export_dir.iterdir()):
        print("-", path.name)

Seed ontology loaded: True
Ontology URI: http://semanticweb.org/STEaMINg/ContextOntology-COInd4
Ontology label: COInd Context Ontology for Industrial Situations
Classes: 47
Properties: 56
[NeoOLAF] Run directory: runs\run_20260323_230608
[NeoOLAF] Pipeline started with 13 layers
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 5.58s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 192.10s
[NeoOLAF] Layer 2/12: layer02_candidate_enrichment

[NeoOLAF] Starting layer: layer02_candidate_enrichment


[NeoOLAF] Finished layer: layer02_candidate_enrichment in 84.71s
[NeoOLAF] Layer 3/12: layer03_candidate_typing_resolution

[NeoOLAF] Starting layer: layer03_candidate_typing_resolution


[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 50.75s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction


[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.01s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


...

[NeoOLAF] Finished layer: layer07_hierarchisation in 0.01s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction


[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.01s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction


[NeoOLAF] Finished layer: layer09_general_axiom_extraction in 0.01s
[NeoOLAF] Layer 10/12: layer10_validation_reasoning

[NeoOLAF] Starting layer: layer10_validation_reasoning


[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.01s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion


[NeoOLAF] Finished layer: layer11_inference_completion in 0.01s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: runs\run_20260323_230608\data\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.11s
[NeoOLAF] Pipeline finished in 399.73s
[NeoOLAF] Total run time: 399.73s
Completion candidates: 0

Export folder:
runs\run_20260323_230608\data\exports

Exported files:
- kg_inferred.json
- kg_inferred.ttl
- kg_local.json
- kg_local.ttl
- ontology_inferred.ttl
- ontology_local.ttl


In [6]:
final_state.reasoning_report.inferred_triples

[CandidateTriple(triple_id='triple_00000', subject_id='cand_e_00002', subject_label='RESET PRG key', subject_type='entity', predicate_id='cand_r_00000', predicate_label='PLC interface', object_id='cand_r_00000', object_label='PLC interface', object_type='relation', chunk_id='chunk_0000', justification="The text states that alarms are divided into two categories, one of which is 'alarms and messages from the PLC interface'.", confidence=0.85, provenance=[Evidence(chunk_id='chunk_0000', chunk_start_char=279, chunk_end_char=292, doc_start_char=279, doc_end_char=292, snippet='intenance manual of the Fanuc series CNC........ 2 alarms and messages from the PLC interface, described below. The messages and alarms emitted by the PLC indicate the detec')]),
 CandidateTriple(triple_id='triple_00001', subject_id='cand_e_00002', subject_label='RESET PRG key', subject_type='entity', predicate_id='cand_r_00001', predicate_label='emitted by', object_id='cand_e_00001', object_label='CNC', object_type='